# Librairies

In [350]:
import re
from collections import deque, defaultdict
from itertools import combinations
from dataclasses import dataclass
import copy
from typing import NamedTuple, Optional

# Classes

In [351]:
class Pair(NamedTuple):
    etage_generator: int = None
    etage_microchip: int = None

    def tolist(self):
        return [self.etage_generator, self.etage_microchip]

In [352]:
@dataclass(frozen=True)
class State:
    etage: int
    list_pairs: tuple[Pair] | list[Pair]

In [353]:
class Item(NamedTuple):
    index_paire: int
    type: str # Microchip ou Generator

In [354]:
@dataclass
class Deplacement:
    index_pair: int 
    new_pair: Pair

# Lecture du fichier

In [355]:
pattern = r"a (.*?)(?=\sa|$)"

In [356]:
with open('real2.txt') as fichier:
    data = [row[:-1] for row in fichier.read().splitlines()]
data

['The first floor contains a promethium generator, a promethium-compatible microchip, a elerium generator, a elerium-compatible microchip, a dilithium generator, and a dilithium-compatible microchip',
 'The second floor contains a cobalt generator, a curium generator, a ruthenium generator, and a plutonium generator',
 'The third floor contains a cobalt-compatible microchip, a curium-compatible microchip, a ruthenium-compatible microchip, and a plutonium-compatible microchip',
 'The fourth floor contains nothing relevant']

In [357]:
def func_clear_row(
    in_row: str
) -> list:

    # Récupère les noms complets des elements de l'étage ex : ['hydrogen-compatible micromicrochip', 'lithium-compatible micromicrochip']
    list_complets_elements = re.findall(pattern, in_row)

    # Si rien a cet etage
    if len(list_complets_elements) == 0:
        return []
        

    # On transforme les noms complets en abreviations 
    # ['hydrogen-compatible micromicrochip', 'lithium-compatible micromicrochip'] ---> ['HM', 'LM']
    list_noms_courts = [
        ''.join([nom_complet_element.split()[0][:2].upper(), nom_complet_element.split()[1][0].upper()]) 
        for nom_complet_element 
        in list_complets_elements
    ]
    
    # On stocke les elements de l'étage dans un dict
    return list_noms_courts

# Traitement

In [358]:
def func_is_valid_list_pairs(
    in_pairs: list[tuple]
) -> bool:

    for pair in in_pairs:
        
        # Si une puce est au même étage que son generateur, elle est protégée
        if pair.etage_microchip == pair.etage_generator:
            continue 

        for pair2 in in_pairs:
            
            # Si une puce se trouve avec un générateur différent et que son générateur a elle n'est pas au même étage, elle grille
            if pair.etage_microchip == pair2.etage_generator:
                return False
    
    return True


In [359]:
def func_get_deplacements_possibles(
    in_state: State
) -> Optional[list[State]]:

    # Stocke les étages auxquels on peut se déplacer (monter d'un etage ou descendre d'un étage)
    possibles_new_etages = [
        new_etage 
        for new_etage 
        in [in_state.etage-1, in_state.etage+1]
        if (new_etage >= 1) and (new_etage <= 4)
    ]

    list_deplacements_possibles: list[Item] = []


    # ------------------ IDENTIFIE LES ELEMENTS DEPLACABLES ------------------ #

    # Pour chaque paire actuelle
    for index_pair, pair in enumerate(in_state.list_pairs):
        # print(f'func_simule_possibilities | Generator: {pair.etage_generator} Microchip: {pair.etage_microchip}')
        
        # Si on a un générateur à l'étage actuel
        if pair.etage_generator == in_state.etage:
                         
            item = Item(
                index_paire = index_pair,
                type = 'generator'
            )

            # On ajoute la nouvelle paire
            list_deplacements_possibles.append(item)

        # Si on a une puce a cet étage
        if pair.etage_microchip == in_state.etage:
            
            item = Item(
                index_paire = index_pair,
                type = 'microchip'
            )

            # On ajoute la nouvelle paire
            list_deplacements_possibles.append(item)

    # Si on a aucun déplacement possible
    if not list_deplacements_possibles:
        return

    # for element_deplacable in list_deplacements_possibles:

        # print(f'func_simule_possibilities | Element déplaçable | {element_deplacable}')


    # ----------------- COMBINAISONS POSSIBLES ENTRE ITEMS A DEPLACER ----------------- #

    list_combinations = (
        list(combinations(list_deplacements_possibles,1)) + 
        list(combinations(list_deplacements_possibles,2))
    )

    # for combinaison in list_combinations:

        # print(f'func_simule_possibilities | Combinaison | {combinaison}')


    # ----------------- COMBINAISONS POSSIBLES + ETAGES POSSIBLES ----------------- #

    # Contiendra les nouveaux States (resultat final)
    list_new_states: list[State] = []

    # Pour chaque nouvel etage possible
    for new_etage in possibles_new_etages:

        # print(f'func_simule_possibilities | Etage {new_etage} | {combinaison}')

        # Pour chaque combinaison possible
        for combo in list_combinations:

                    
            # Utilise une copie pour ne pas modifier l'original
            list_new_pairs : list[Pair] = [pair.tolist() for pair in in_state.list_pairs]

            # print(f'func_simule_possibilities | Etage {new_etage} | Combo {combo}')
            
            # Pour chaque item du combo
            for item in combo:
                # print(f'func_simule_possibilities | Etage {new_etage} | Combo {combo} | Item {item} | Ancien {list_new_pairs[item.index_paire]}')

                # Si l'item est un générateur
                if item.type == "generator":
                
                    list_new_pairs[item.index_paire][0] = new_etage

                # Si l'item est une microchip
                if item.type == "microchip":
                
                    list_new_pairs[item.index_paire][1] = new_etage

                # print(f'func_simule_possibilities | Etage {new_etage} | Combo {combo} | Item {item} | Nouveau {list_new_pairs[item.index_paire]}')

            
           
            # Reconversion de la liste de liste ----> tuple de tuple només
            list_new_pairs = tuple( 
                    Pair(
                        etage_generator = couple[0],
                        etage_microchip = couple[1] 
                    )
                    for couple 
                    in sorted(list_new_pairs)
                )
            
            # Si la liste de pairs est invalide
            if not func_is_valid_list_pairs(in_pairs=list_new_pairs):
                continue
            
            
            # Construction de la liste des nouveaux states
            list_new_states.append(
                State(
                    etage = new_etage,
                    list_pairs = list_new_pairs
                )
            )    
            

    # print(f'func_simule_possibilities | list_deplacements_possibles | {list_new_states}\n')

    return list_new_states

In [360]:
dict_etages = {
    num_etage: func_clear_row(row) 
    for num_etage, row 
    in zip(range(1,len(data)),data) 
}

# On rajoute l'étapge 4
dict_etages[4] = []

dict_etages

{1: ['PRG', 'PRM', 'ELG', 'ELM', 'DIG', 'DIM'],
 2: ['COG', 'CUG', 'RUG', 'PLG'],
 3: ['COM', 'CUM', 'RUM', 'PLM'],
 4: []}

In [361]:
dict_pairs = defaultdict(lambda:['',''])
for etage, list_elements in dict_etages.items():
    for element in list_elements:

        # Si c'est un generateur
        if element[2] == 'G':

            # On le met en premiere position du tuple
            dict_pairs[element[:2]][0] = etage

        # Si c'est une microchip, on le met en seconde position du tuple
        else:
            dict_pairs[element[:2]][1] = etage

dict_pairs

defaultdict(<function __main__.<lambda>()>,
            {'PR': [1, 1],
             'EL': [1, 1],
             'DI': [1, 1],
             'CO': [2, 3],
             'CU': [2, 3],
             'RU': [2, 3],
             'PL': [2, 3]})

In [362]:
# Transforme les le dictionnaire de listes en tuples de tuples

dict_pairs = tuple(
    Pair(
        etage_generator = paire[0],
        etage_microchip = paire[1]
    ) 
    for paire 
    in dict_pairs.values()
    )
dict_pairs

(Pair(etage_generator=1, etage_microchip=1),
 Pair(etage_generator=1, etage_microchip=1),
 Pair(etage_generator=1, etage_microchip=1),
 Pair(etage_generator=2, etage_microchip=3),
 Pair(etage_generator=2, etage_microchip=3),
 Pair(etage_generator=2, etage_microchip=3),
 Pair(etage_generator=2, etage_microchip=3))

In [363]:
start_state = State(    
    etage = 1,
    list_pairs = tuple(sorted(dict_pairs))
)

visited: set[State] = {start_state}
queue: deque[tuple[int, State]] = deque([(0,start_state)])

while queue:
    # print(f'Queue: {queue}')
    etape, state = queue.popleft()
    # print(f"| - - - - - - - - - - - - - - - - - - - - - - - - Etape {etape} - - - - - - - - - - - - - - - - - - - - - - - - |\n")
    # print(f'Etage {state.etage}')
    # for num_pair, pair in enumerate(state.list_pairs):
        # print(f'Paire {num_pair} | Generator: {pair.etage_generator} Microchip: {pair.etage_microchip}')
        

    # Si on a tout au dernier étage
    if all(pair==(4,4) for pair in state.list_pairs):
        print('fin')
        print(etape)
        break

    # On simule les possibilités
    list_new_states: list[State] = func_get_deplacements_possibles(
        in_state = state
    )

    # On aucune possibilité de déplacement
    if not list_new_states:
        continue

    # On ajoute les possibilités à la file
    for new_state in list_new_states:      

        # Vérifie si ce state n'a pas déja été testé
        if new_state not in visited:
            
            # Ajoute le nouvel état a la file (+ le numéro de l'étape)
            queue.append([etape+1, new_state])
            
            # Ajout du nouvel état à l'ensemble de états testés
            visited.add(new_state)


fin
57
